# Week 48, Day 5: Predictive coding: perception as explaining away the input

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w048/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Finding the causes

Write infer(X, U): start from zero causes; for each step compute the error E = X - U R, move R by eta U^T E, then soft-threshold by eta lambda. Return the causes and the error. Run it on one patch with the true dictionary B and record the error norm at each step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1999)
P = 8; bars = []
for i in range(P):
    h = np.zeros((P, P)); h[i] = 1; bars.append(h.ravel())
    v = np.zeros((P, P)); v[:, i] = 1; bars.append(v.ravel())
B = np.array(bars).T; B /= np.linalg.norm(B, axis=0)          # 64 pixels x 16 causes

def sample(n):
    C = np.zeros((16, n))
    for j in range(n):
        C[r.choice(16, 2, replace=False), j] = r.uniform(0.5, 1.5, 2)
    return B @ C + 0.05 * r.normal(size=(64, n)), C

def infer(X, U, steps=40, eta=0.1, lam=0.1, record=False):
    R = np.zeros((U.shape[1], X.shape[1])); hist = []
    for _ in range(steps):
        ...
        if record:
            hist.append(np.linalg.norm(X - U @ R))
    return R, X - U @ R, hist

x, c = sample(1); true_idx = np.where(c[:, 0] > 0)[0]
R, E, err_hist = infer(x, B, record=True); r_hat = R[:, 0]
print(f"true causes {sorted(true_idx.tolist())}, largest inferred {sorted(np.argsort(np.abs(r_hat))[-2:].tolist())}; error {err_hist[0]:.2f} -> {err_hist[-1]:.2f} (input norm {np.linalg.norm(x):.2f})")
plt.plot(err_hist); plt.xlabel("inference step"); plt.ylabel("|prediction error|"); plt.show()

In [ ]:
import numpy as np
def test_error():
    "the prediction error falls below 35% of the input's norm, from above 80%"
    assert err_hist[-1] < 0.35 * np.linalg.norm(x) and err_hist[0] > 0.8 * np.linalg.norm(x)
def test_causes():
    "the two largest inferred causes are the two true bars"
    assert set(np.argsort(np.abs(r_hat))[-2:]) == set(true_idx)

test_error()
test_causes()
print("All checks passed")

### Exercise: Discovering the bars

Start from random unit-norm columns; for 400 batches of 50 patches, infer causes, update U by 1.0 E R^T / 50 and renormalise columns. Then compute, for each true bar, the best |cosine| with any learned column, and the relative error on 200 new patches.

In [ ]:
U = r.normal(size=(64, 16)); U /= np.linalg.norm(U, axis=0)
for it in range(400):
    X, _ = sample(50)
    R, E, _ = infer(X, U)
    ...
match = np.abs(B.T @ U).max(1)
Xt, _ = sample(200); _, Et, _ = infer(Xt, U)
rel_err = np.linalg.norm(Et) / np.linalg.norm(Xt)
print(f"best match per true bar: min {match.min():.2f}, mean {match.mean():.2f}; relative error {rel_err:.3f}")
fig, ax = plt.subplots(2, 8, figsize=(8, 2.2))
for k, a_ in enumerate(ax.ravel()):
    a_.imshow(U[:, k].reshape(8, 8), cmap="gray"); a_.axis("off")
plt.show()

In [ ]:
import numpy as np
def test_dictionary():
    "every true bar is matched by a learned column with |cosine| above 0.9"
    assert (match > 0.9).all()
def test_error():
    "after learning, the relative prediction error on new patches is below 0.35"
    assert rel_err < 0.35

test_dictionary()
test_error()
print("All checks passed")

### Exercise: Repetition suppression and mismatch

Make a sequence of 200 patches: the standard (bars 0 and 5) on 90% of trials and a deviant (bars 2 and 11) on 10% (never in the first 20). Run the prediction update with alpha 0.3 and record the error norm at each trial.

In [ ]:
std = B[:, [0, 5]].sum(1); dev = B[:, [2, 11]].sum(1)
is_dev = (r.random(200) < 0.1); is_dev[:20] = False
seq = np.where(is_dev[:, None], dev, std) + 0.05 * r.normal(size=(200, 64))
p = np.zeros(64); err = np.zeros(200); alpha = 0.3
for t in range(200):
    ...
std_idx, dev_idx = np.where(~is_dev)[0], np.where(is_dev)[0]
print(f"error to first standard {err[std_idx[0]]:.2f}, to the 11th {err[std_idx[10]]:.2f}; late standards {err[std_idx[20:]].mean():.2f}, deviants {err[dev_idx].mean():.2f}")
plt.plot(err, "k-", lw=0.6); plt.plot(dev_idx, err[dev_idx], "ro", label="deviant"); plt.xlabel("trial"); plt.ylabel("|prediction error|"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_suppression():
    "the error to the standard falls over its first repetitions"
    assert err[std_idx[0]] > 3 * err[std_idx[10]]
def test_mismatch():
    "deviants evoke at least three times the error of late standards"
    assert err[dev_idx].mean() > 3 * err[std_idx[20:]].mean()

test_suppression()
test_mismatch()
print("All checks passed")